# 8주차 예제 — 문제 정의하는 법 + 외부데이터 결합 가이드

이번 노트북에서는 특정 데이터셋의 파이프라인을 처음부터 끝까지 따라가기보다, **미니 프로젝트를 스스로 설계할 때 필요한 두 가지 요소**인 문제 정의와 외부데이터 결합 점검을 짧은 예제로 함께 살펴봅니다.

1. 문제 정의 체크리스트
2. 외부데이터 결합 점검 ① — 키 매칭
3. 외부데이터 결합 점검 ② — 단위 통일
4. 지금까지 배운 조인 사례 되짚기
5. 미니 프로젝트 설계 체크리스트

## 제공 트랙 데이터

8주차 과제에서 제공 트랙을 선택한다면 아래 원본 페이지에서 데이터 설명과 이용 조건을 확인하고 파일을 내려받을 수 있습니다. 링크는 `README.md`에 안내된 주소와 같습니다.

- [제주도교통량](https://dacon.io/competitions/official/235985)
- [신용카드세그먼트](https://dacon.io/competitions/official/236460)

다운로드 날짜와 사용한 파일명을 기록해 두면 분석을 다시 실행할 때 도움이 됩니다.


## Part 1. 문제 정의 체크리스트

데이터를 열기 전에 아래 네 가지 질문을 함께 생각해 봅니다. 먼저 답을 정리해 두면 코드를 작성하는 동안에도 분석의 방향을 편안하게 확인할 수 있습니다.

1. **무엇이 궁금합니까?** — 한 문장 질문으로 정리합니다. (예: "어떤 지역이 대중교통 접근성이 낮습니까?")
2. **그 질문에 답하려면 어떤 데이터가 필요합니까?** — 필요한 데이터를 실제로 확보할 수 있는지도 확인합니다.
3. **완료를 어떻게 판단합니까?** — 숫자, 목록, 지도 중 어떤 형태로 결론을 제시할지 정합니다.
4. **발표에서 무엇을 보여줍니까?** — 그래프 1~2개로 청중에게 결론을 전달할 수 있는지 살펴봅니다.


In [ ]:
import pandas as pd


## Part 2. 외부데이터 결합 점검 ① — 키 매칭

서로 다른 곳에서 온 두 테이블을 합칠 때 ID 형식이 미묘하게 다르면, 오류 메시지는 없지만 일부 행만 연결되고 나머지 값은 `NaN`으로 채워질 수 있습니다. 따라서 조인 뒤에는 연결된 행 수와 `NaN` 개수를 함께 확인하면 좋습니다.


In [ ]:
customers = pd.DataFrame({'고객ID': ['A001', 'A002', 'A003'], '이름': ['김민준', '이서연', '박도윤']})
orders = pd.DataFrame({'고객ID': ['a001', 'A002', 'A-003'], '금액': [10000, 20000, 15000]})

merged = customers.merge(orders, on='고객ID', how='left')
merged


3명 중 1명(`A002`)만 의도대로 연결되었습니다. `a001`(소문자), `A-003`(하이픈)처럼 표기가 다르면 서로 다른 값으로 처리되기 때문입니다. `merge` 자체는 오류 없이 실행되므로, **결과의 `NaN` 개수와 연결된 행 수를 확인하면 이러한 표기 차이를 발견할 수 있습니다.**


In [ ]:
def normalize_id(s):
    return s.str.upper().str.replace('-', '', regex=False)

customers['고객ID_정규화'] = normalize_id(customers['고객ID'])
orders['고객ID_정규화'] = normalize_id(orders['고객ID'])

merged_fixed = customers.merge(orders, on='고객ID_정규화', how='left')
merged_fixed


키를 대문자로 통일하고 하이픈을 제거한 뒤 조인하면 3명 모두 의도대로 연결됩니다. **조인 전에 양쪽 키의 형식(대소문자, 구분자, 공백)이 같은지 먼저 살펴보면** 연결 오류를 줄일 수 있습니다.


## Part 3. 외부데이터 결합 점검 ② — 단위 통일

키가 잘 맞더라도 단위가 다르면 의도와 다른 계산 결과가 나올 수 있습니다.


In [ ]:
sales = pd.DataFrame({'상품': ['A', 'B'], '매출_원': [1500000, 2300000]})
budget = pd.DataFrame({'상품': ['A', 'B'], '예산_만원': [120, 200]})

m = sales.merge(budget, on='상품')
m['잘못된_비율'] = m['예산_만원'] / m['매출_원']  # 단위를 통일하지 않은 비교 예시
m


In [ ]:
m['예산_원'] = m['예산_만원'] * 10000  # 단위를 먼저 맞춘다
m['올바른_비율'] = m['예산_원'] / m['매출_원']
m[['상품', '매출_원', '예산_원', '잘못된_비율', '올바른_비율']]


코드에서 계산한 `'잘못된_비율'`과 `'올바른_비율'`은 10,000배 차이가 납니다. 단위가 서로 다른 상태에서도 코드는 오류 없이 실행되고 숫자를 반환합니다. 따라서 **계산 전에 단위를 통일하고, 결과가 정의상 가능한 범위에 있는지 확인하면** 이 차이를 일찍 발견할 수 있습니다. 예를 들어 비율이라면 먼저 0~1 값인지, 0~100% 값인지 기준을 정합니다.


## Part 4. 지금까지 배운 조인 사례 되짚기

이번 학기에 다룬 사례에도 비슷한 점검 지점이 있었습니다.

| 주차 | 조인/결합 | 만난 문제 | 해결 |
|---|---|---|---|
| 3주차 | 전력사용량 + 건물정보 | 결측치가 `NaN`이 아니라 문자열 `'-'`로 표시됨 | `'-'`를 0으로 명시적으로 치환 후 형변환 |
| 4주차 | 물류 격자ID/카테고리 인코딩 | test에 train에서 못 본 격자ID가 다수 | train에만 `fit`한 `OneHotEncoder(handle_unknown='ignore')`를 Pipeline으로 사용하고 미지 ID를 따로 점검 |
| 6주차 | 이커머스 5개 테이블 | `discount`의 `월` 컬럼이 `'Jan'` 같은 축약형, `sales`는 날짜 전체 | `dt.strftime('%b')`로 형식을 맞춰서 조인 |

세 사례를 살펴보면, **같은 의미를 서로 다르게 표기한 경우**가 조인 결과가 기대와 달라지는 주요 원인 가운데 하나임을 알 수 있습니다. 미니 프로젝트에서 외부데이터를 더할 때도 두 테이블의 키 형식과 숫자 컬럼의 단위가 같은지 차근차근 확인해 봅니다.


## Part 5. 미니 프로젝트 설계 체크리스트

과제(`assignment_baseline.ipynb`)를 시작하기 전에 아래 항목을 하나씩 정리해 봅니다.

- [ ] Part 1의 질문 4개에 한 문장씩 답했습니다.
- [ ] 사용할 데이터를 정하고, 조인이 필요하다면 키와 단위를 미리 확인했습니다.
- [ ] 최종적으로 보여줄 그래프나 표를 1~2개 정도 미리 그려 보았습니다. 간단한 스케치도 충분합니다.
- [ ] 발표 시간과 질의응답을 고려해 분량을 조절했습니다.
